# 01 — Raw Data Inventory & Schema Analysis**Project:** Surgery Duration Prediction — Hillel Yaffe Medical Center  **Stage:** Exploratory inventory (pre-ETL)  **Constraint:** No merging, cleaning, imputation, or feature engineering.This notebook performs a complete inventory of all raw datasets under `data/raw/` to understand structure, keys, relationships, and merge risks before building the ETL pipeline.

## 1. Setup & Configuration

In [ ]:
import warnings
from pathlib import Path
from collections import defaultdict
from itertools import combinations

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RAW_DIR = Path("data/raw")
UNIQUENESS_PK_THRESHOLD = 0.95  # ratio of unique/total for PK candidacy
HIGH_UNIQUENESS_THRESHOLD = 0.90

# Known column name aliases for cross-table join detection
CASE_ID_ALIASES = {"מספר מקרה", "מקרה"}
PATIENT_ID_ALIASES = {"patient", "Patient", "Medical Record"}

# Headerless lab CSV column names (file has no header row)
LAB_CSV_COLUMNS = [
    "Patient",
    "מספר מקרה",
    "קטגוריית בדיקה",
    "שם בדיקה",
    "ערך",
    "תאריך בדיקה",
]

# Human-readable dataset descriptions (inferred from filenames)
DATASET_DESCRIPTIONS = {
    "ניתוחים_260415": "Surgery cases — procedures, timestamps, staff, anesthesia (likely central fact table)",
    "בדיקות מעבדה": "Pre-operative laboratory test results (long format, multiple tests per case)",
    "הרדמה": "Anesthesia type per surgery case",
    "זמן חתימת מנתח על sign in": "Surgeon electronic sign-in timestamp per case",
    "זמן חתימת מנתח על sign out": "Surgeon electronic sign-out timestamp per case",
    "לחץ דם": "Pre-operative blood pressure (systolic/diastolic) per case",
    "מחלות רקע": "Patient background diseases (ICD-9 codes, patient-level)",
    "סוג דם גובה משקל BMI": "Patient vitals: blood type, height, weight, BMI per case",
    "סטורציה": "Pre-operative oxygen saturation per case",
    "עישון": "Smoking status per surgery case",
    "צוות סיעודי בניתוח": "Scrub nurse assignments per case (may be multi-valued)",
    "רגישות אחרת": "Other allergies/sensitivities (patient-level)",
    "רגישות לתרופות": "Drug allergies/sensitivities (patient-level)",
    "תרופות בניתוח": "Medications administered during surgery (long format)",
    "תרופות קבועות": "Chronic/home medications (patient-level)",
}

print(f"Raw data directory: {RAW_DIR.resolve()}")
print(f"Files found: {len(list(RAW_DIR.iterdir()))}")


## 2. Helper FunctionsReusable analysis utilities for per-dataset and cross-table inventory.

In [ ]:
def load_raw_dataset(file_path: Path) -> pd.DataFrame:
    """Load a single raw file with format-specific handling."""
    stem = file_path.stem
    if file_path.suffix.lower() == ".csv":
        df = pd.read_csv(
            file_path,
            header=None,
            names=LAB_CSV_COLUMNS,
            encoding="utf-8-sig",
            low_memory=False,
        )
    else:
        df = pd.read_excel(file_path)
    df.attrs["source_file"] = file_path.name
    df.attrs["dataset_name"] = stem
    return df


def normalize_col_name(col: str) -> str:
    return str(col).strip().lower()


def schema_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Detailed column-level schema summary."""
    rows = []
    n = len(df)
    for col in df.columns:
        s = df[col]
        null_count = int(s.isna().sum())
        non_null = n - null_count
        nunique = int(s.nunique(dropna=True))
        rows.append({
            "column": col,
            "dtype": str(s.dtype),
            "non_null_count": non_null,
            "null_count": null_count,
            "null_pct": round(100 * null_count / n, 2) if n else 0,
            "n_unique": nunique,
            "uniqueness_ratio": round(nunique / non_null, 4) if non_null else np.nan,
        })
    return pd.DataFrame(rows).sort_values("column").reset_index(drop=True)


def is_likely_date_column(col: str) -> bool:
    """Heuristic: exclude timestamp columns from PK candidacy."""
    date_keywords = ["תאריך", "מועד", "date", "time", "sign in", "sign out", "entry_date", "זמן", "שעת"]
    col_str = str(col).lower()
    return any(kw in col_str or kw in str(col) for kw in date_keywords)


def detect_id_columns(df: pd.DataFrame, schema: pd.DataFrame) -> pd.DataFrame:
    """Identify potential identifier / primary-key columns."""
    records = []
    for _, row in schema.iterrows():
        col = row["column"]
        norm = normalize_col_name(col)
        is_date_col = is_likely_date_column(col)
        name_match = (
            norm.endswith("_id")
            or norm.endswith(" id")
            or col in CASE_ID_ALIASES
            or col in PATIENT_ID_ALIASES
        )
        reasons = []
        if name_match:
            reasons.append("name_pattern")
        if row["uniqueness_ratio"] >= HIGH_UNIQUENESS_THRESHOLD and not is_date_col:
            reasons.append("high_uniqueness")
        if row["uniqueness_ratio"] == 1.0 and row["null_count"] == 0 and not is_date_col:
            reasons.append("unique_no_nulls")
        if reasons:
            pk_candidate = (
                name_match
                and row["null_count"] == 0
                and row["uniqueness_ratio"] >= UNIQUENESS_PK_THRESHOLD
            )
            records.append({
                "column": col,
                "n_unique": row["n_unique"],
                "uniqueness_ratio": row["uniqueness_ratio"],
                "null_count": row["null_count"],
                "reasons": ", ".join(reasons),
                "pk_candidate": pk_candidate,
            })
    return pd.DataFrame(records).sort_values(["pk_candidate", "uniqueness_ratio"], ascending=[False, False]).reset_index(drop=True)


def check_data_quality(df: pd.DataFrame, id_cols: pd.DataFrame) -> dict:
    """Duplicate row and PK-candidate duplicate checks."""
    dup_rows = int(df.duplicated().sum())
    pk_dup_details = []
    if not id_cols.empty:
        for col in id_cols.loc[id_cols["pk_candidate"], "column"]:
            dup_vals = int(df[col].duplicated().sum() - df[col].isna().sum())
            dup_count = int(df[col].duplicated(keep=False).sum())
            pk_dup_details.append({"column": col, "rows_in_duplicate_groups": dup_count, "is_unique": dup_count == 0})
    return {"duplicate_rows": dup_rows, "pk_duplicate_checks": pd.DataFrame(pk_dup_details)}


def identify_date_columns(df: pd.DataFrame) -> list:
    """Auto-detect date/datetime columns by name and parseability."""
    date_keywords = ["תאריך", "מועד", "date", "time", "sign in", "sign out", "entry_date", "זמן", "שעת"]
    candidates = []
    for col in df.columns:
        col_lower = str(col).lower()
        if any(kw in col_lower or kw in str(col) for kw in date_keywords):
            candidates.append(col)
            continue
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            candidates.append(col)
            continue
        if df[col].dtype == object:
            sample = df[col].dropna().head(200)
            if len(sample) == 0:
                continue
            parsed = pd.to_datetime(sample, errors="coerce", dayfirst=True)
            if parsed.notna().mean() >= 0.8:
                candidates.append(col)
    return list(dict.fromkeys(candidates))


def analyze_date_columns(df: pd.DataFrame, date_cols: list) -> pd.DataFrame:
    if not date_cols:
        return pd.DataFrame()
    rows = []
    for col in date_cols:
        s = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
        valid = s.dropna()
        rows.append({
            "column": col,
            "min_date": valid.min(),
            "max_date": valid.max(),
            "n_unique_dates": int(valid.dt.normalize().nunique()) if len(valid) else 0,
            "null_count": int(s.isna().sum()),
            "parseable_pct": round(100 * valid.shape[0] / len(df), 2),
        })
    return pd.DataFrame(rows)


def analyze_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not num_cols:
        return pd.DataFrame()
    desc = df[num_cols].describe().T
    desc = desc.rename(columns={"50%": "median"})
    desc["column"] = desc.index
    return desc[["column", "count", "mean", "std", "min", "median", "max"]].reset_index(drop=True)


def analyze_categorical_columns(df: pd.DataFrame, max_cardinality: int = 500) -> dict:
    """Top-20 value counts for low/medium cardinality object columns."""
    results = {}
    for col in df.select_dtypes(include=["object", "string"]).columns:
        nunique = df[col].nunique(dropna=True)
        if nunique > max_cardinality:
            results[col] = {
                "n_unique": nunique,
                "note": f"High cardinality ({nunique} categories) — showing top 20 only",
                "top_20": df[col].value_counts(dropna=False).head(20),
            }
        else:
            results[col] = {
                "n_unique": nunique,
                "note": None,
                "top_20": df[col].value_counts(dropna=False).head(20),
            }
    return results


def get_standardized_key_columns(df: pd.DataFrame) -> dict:
    """Map dataset columns to semantic roles for cross-table analysis."""
    mapping = {"case_id": None, "patient_id": None, "medical_record": None}
    for col in df.columns:
        norm = normalize_col_name(col)
        if col in CASE_ID_ALIASES or norm in {normalize_col_name(x) for x in CASE_ID_ALIASES}:
            mapping["case_id"] = col
        if col in PATIENT_ID_ALIASES or norm in {normalize_col_name(x) for x in PATIENT_ID_ALIASES}:
            if norm == "medical record":
                mapping["medical_record"] = col
            else:
                mapping["patient_id"] = col
    return mapping


def classify_relationship(n_a: int, u_a: int, n_b: int, u_b: int, overlap: int) -> str:
    """Classify join relationship based on uniqueness and overlap."""
    if overlap == 0:
        return "No overlap"
    left_unique = u_a == n_a
    right_unique = u_b == n_b
    if left_unique and right_unique:
        return "One-to-One (1:1)"
    if left_unique and not right_unique:
        return "One-to-Many (1:N)"
    if not left_unique and right_unique:
        return "Many-to-One (N:1)"
    return "Many-to-Many (N:N)"


def analyze_dataset(file_path: Path) -> dict:
    """Full per-dataset inventory analysis."""
    df = load_raw_dataset(file_path)
    name = file_path.stem
    schema = schema_summary(df)
    id_cols = detect_id_columns(df, schema)
    quality = check_data_quality(df, id_cols)
    date_cols = identify_date_columns(df)
    date_analysis = analyze_date_columns(df, date_cols)
    numeric_analysis = analyze_numeric_columns(df)
    cat_analysis = analyze_categorical_columns(df)
    keys = get_standardized_key_columns(df)

    pk_candidates = id_cols.loc[id_cols["pk_candidate"], "column"].tolist() if not id_cols.empty else []

    return {
        "name": name,
        "file": file_path.name,
        "description": DATASET_DESCRIPTIONS.get(name, "—"),
        "df": df,
        "n_rows": len(df),
        "n_cols": len(df.columns),
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1e6, 2),
        "schema": schema,
        "id_columns": id_cols,
        "pk_candidates": pk_candidates,
        "quality": quality,
        "date_cols": date_cols,
        "date_analysis": date_analysis,
        "numeric_analysis": numeric_analysis,
        "categorical_analysis": cat_analysis,
        "keys": keys,
    }


## 3. Load All Raw Datasets

In [ ]:
raw_files = sorted(
    [p for p in RAW_DIR.iterdir() if p.suffix.lower() in {".csv", ".xlsx", ".xls"}],
    key=lambda p: p.name,
)

print("Files to analyze:")
for f in raw_files:
    print(f"  • {f.name}")

datasets = {}
load_errors = {}

for f in raw_files:
    try:
        print(f"Loading {f.name}...", end=" ")
        datasets[f.stem] = analyze_dataset(f)
        print(f"OK — {datasets[f.stem]['n_rows']:,} rows × {datasets[f.stem]['n_cols']} cols")
    except Exception as e:
        load_errors[f.name] = str(e)
        print(f"FAILED — {e}")

if load_errors:
    print("\n⚠️ Load errors:")
    for k, v in load_errors.items():
        print(f"  {k}: {v}")
else:
    print(f"\n✓ Successfully loaded {len(datasets)} datasets")


## 4. Dataset-by-Dataset AnalysisFor each raw file: schema, identifiers, data quality, dates, numerics, and categoricals.

In [ ]:
for name, info in datasets.items():
    display(pd.DataFrame([{
        "Dataset": name,
        "File": info["file"],
        "Description": info["description"],
        "Rows": f"{info['n_rows']:,}",
        "Columns": info["n_cols"],
        "Memory (MB)": info["memory_mb"],
        "PK Candidates": ", ".join(info["pk_candidates"]) or "—",
        "Case ID Col": info["keys"]["case_id"] or "—",
        "Patient ID Col": info["keys"]["patient_id"] or "—",
    }]))


In [ ]:
for name, info in datasets.items():
    print("=" * 90)
    print(f"DATASET: {name}")
    print(f"File: {info['file']}  |  Rows: {info['n_rows']:,}  |  Cols: {info['n_cols']}  |  Memory: {info['memory_mb']} MB")
    print(f"Description: {info['description']}")
    print("=" * 90)

    print("\n--- Schema Summary ---")
    display(info["schema"])

    print("\n--- Potential Identifier Columns ---")
    if info["id_columns"].empty:
        print("No strong identifier columns detected.")
    else:
        display(info["id_columns"])

    print("\n--- Data Quality ---")
    print(f"Duplicate rows (full row): {info['quality']['duplicate_rows']:,}")
    pk_dup = info["quality"]["pk_duplicate_checks"]
    if not pk_dup.empty:
        display(pk_dup)
    else:
        print("No PK-candidate columns to check for duplicates.")

    print("\n--- Date Column Analysis ---")
    if info["date_analysis"].empty:
        print("No date columns detected.")
    else:
        display(info["date_analysis"])

    print("\n--- Numeric Column Statistics ---")
    if info["numeric_analysis"].empty:
        print("No numeric columns.")
    else:
        display(info["numeric_analysis"])

    print("\n--- Categorical Column Analysis (Top 20) ---")
    if not info["categorical_analysis"]:
        print("No categorical (object) columns.")
    else:
        for col, cat_info in info["categorical_analysis"].items():
            print(f"\n  Column: {col}  |  Unique categories: {cat_info['n_unique']:,}")
            if cat_info["note"]:
                print(f"  Note: {cat_info['note']}")
            display(cat_info["top_20"].to_frame("count"))

    print("\n--- Sample Rows (first 3) ---")
    display(info["df"].head(3))

    print("\n")


## 5. Global Inventory Report

In [ ]:
inventory_rows = []
for name, info in datasets.items():
    inventory_rows.append({
        "dataset": name,
        "file": info["file"],
        "row_count": info["n_rows"],
        "column_count": info["n_cols"],
        "memory_mb": info["memory_mb"],
        "candidate_primary_keys": ", ".join(info["pk_candidates"]) or "—",
        "case_id_column": info["keys"]["case_id"] or "—",
        "patient_id_column": info["keys"]["patient_id"] or "—",
        "medical_record_column": info["keys"]["medical_record"] or "—",
        "duplicate_rows": info["quality"]["duplicate_rows"],
        "description": info["description"],
    })

global_inventory = pd.DataFrame(inventory_rows).sort_values("row_count", ascending=False).reset_index(drop=True)

print("GLOBAL INVENTORY")
display(global_inventory)


## 6. Cross-Table AnalysisIdentify shared columns, overlapping key values, and suggested relationships.

In [ ]:
# 6.1 Columns appearing in multiple datasets
column_occurrence = defaultdict(list)
for name, info in datasets.items():
    for col in info["df"].columns:
        column_occurrence[str(col)].append(name)

shared_columns = {
    col: tables for col, tables in column_occurrence.items() if len(tables) > 1
}

shared_cols_df = pd.DataFrame([
    {"column": col, "n_datasets": len(tables), "datasets": ", ".join(sorted(tables))}
    for col, tables in sorted(shared_columns.items(), key=lambda x: -len(x[1]))
])

print("Columns appearing in multiple datasets:")
display(shared_cols_df)


In [ ]:
# 6.2 Semantic key registry for relationship analysis
key_registry = {}
for name, info in datasets.items():
    key_registry[name] = {
        "case_id": info["keys"]["case_id"],
        "patient_id": info["keys"]["patient_id"],
        "medical_record": info["keys"]["medical_record"],
    }

key_registry_df = pd.DataFrame(key_registry).T
print("Standardized key column mapping per dataset:")
display(key_registry_df)


In [ ]:
# 6.3 Pairwise relationship detection on shared / aliased key columns

def get_column_series(df, col):
    s = df[col].dropna()
    if pd.api.types.is_numeric_dtype(s):
        return s.astype("Int64").astype(str)
    return s.astype(str).str.strip()


def compute_pairwise_relationships(datasets: dict) -> pd.DataFrame:
    relationships = []

    # Build list of (dataset, semantic_role, column_name)
    key_columns = []
    for ds_name, info in datasets.items():
        for role, col in info["keys"].items():
            if col:
                key_columns.append((ds_name, role, col))

    # Also compare columns with identical names across tables
    for col_name, ds_list in shared_columns.items():
        for ds_a, ds_b in combinations(sorted(ds_list), 2):
            key_columns.append((ds_a, f"shared:{col_name}", col_name))
            key_columns.append((ds_b, f"shared:{col_name}", col_name))

    # Deduplicate pairs
    seen_pairs = set()
    for i, (ds_a, role_a, col_a) in enumerate(key_columns):
        for ds_b, role_b, col_b in key_columns[i + 1:]:
            if ds_a == ds_b:
                continue
            pair_key = tuple(sorted([(ds_a, col_a), (ds_b, col_b)]))
            if pair_key in seen_pairs:
                continue
            seen_pairs.add(pair_key)

            df_a = datasets[ds_a]["df"]
            df_b = datasets[ds_b]["df"]
            if col_a not in df_a.columns or col_b not in df_b.columns:
                continue

            s_a = get_column_series(df_a, col_a)
            s_b = get_column_series(df_b, col_b)
            set_a = set(s_a.unique())
            set_b = set(s_b.unique())
            overlap = set_a & set_b

            if not overlap:
                continue

            n_a, u_a = len(s_a), len(set_a)
            n_b, u_b = len(s_b), len(set_b)
            rel_type = classify_relationship(n_a, u_a, n_b, u_b, len(overlap))

            # Suggest relationship label
            if "case_id" in role_a or "case_id" in role_b or "מקרה" in col_a or "מקרה" in col_b:
                suggested = "Surgery case linkage"
            elif "patient_id" in role_a or "patient_id" in role_b:
                suggested = "Patient linkage"
            elif "medical_record" in role_a or "medical_record" in role_b:
                suggested = "Medical record linkage"
            else:
                suggested = "Shared attribute"

            relationships.append({
                "table_a": ds_a,
                "column_a": col_a,
                "table_b": ds_b,
                "column_b": col_b,
                "overlap_count": len(overlap),
                "overlap_pct_a": round(100 * len(overlap) / max(u_a, 1), 2),
                "overlap_pct_b": round(100 * len(overlap) / max(u_b, 1), 2),
                "rows_a": n_a,
                "unique_a": u_a,
                "rows_b": n_b,
                "unique_b": u_b,
                "relationship_type": rel_type,
                "suggested_relationship": suggested,
            })

    rel_df = pd.DataFrame(relationships)
    if rel_df.empty:
        return rel_df
    return rel_df.sort_values(["suggested_relationship", "overlap_count"], ascending=[True, False]).reset_index(drop=True)


relationship_report = compute_pairwise_relationships(datasets)

print("RELATIONSHIP REPORT")
print("=" * 90)
if relationship_report.empty:
    print("No overlapping key relationships detected.")
else:
    display(relationship_report[[
        "table_a", "column_a", "table_b", "column_b",
        "suggested_relationship", "relationship_type",
        "overlap_count", "overlap_pct_a", "overlap_pct_b",
    ]])


In [ ]:
# 6.4 Case-level granularity check (critical for merge planning)
case_granularity = []
for name, info in datasets.items():
    case_col = info["keys"]["case_id"]
    if case_col:
        df = info["df"]
        n_rows = len(df)
        n_cases = df[case_col].nunique(dropna=True)
        rows_per_case = round(n_rows / max(n_cases, 1), 2)
        case_granularity.append({
            "dataset": name,
            "case_id_column": case_col,
            "n_rows": n_rows,
            "n_unique_cases": n_cases,
            "avg_rows_per_case": rows_per_case,
            "granularity": "1 row/case" if rows_per_case == 1 else f"~{rows_per_case} rows/case (multi-record)",
        })

case_granularity_df = pd.DataFrame(case_granularity).sort_values("avg_rows_per_case", ascending=False)
print("Case-level granularity (important for join duplication risk):")
display(case_granularity_df)


In [ ]:
# 6.5 Patient-level tables without case ID (require careful join path)
patient_only_tables = []
for name, info in datasets.items():
    if info["keys"]["patient_id"] and not info["keys"]["case_id"]:
        df = info["df"]
        patient_col = info["keys"]["patient_id"]
        patient_only_tables.append({
            "dataset": name,
            "patient_id_column": patient_col,
            "n_rows": len(df),
            "n_unique_patients": df[patient_col].nunique(dropna=True),
            "avg_records_per_patient": round(len(df) / max(df[patient_col].nunique(), 1), 2),
            "join_risk": "Patient-level only — joining directly to surgeries may duplicate or require aggregation",
        })

if patient_only_tables:
    print("Patient-level tables (no case ID):")
    display(pd.DataFrame(patient_only_tables))
else:
    print("All tables have case-level keys.")


## 7. Merge Planning ReportBased on schema discovery — **no merges performed in this notebook**.

In [ ]:
# Identify likely central table
CENTRAL_TABLE_CANDIDATE = "ניתוחים_260415"

central = datasets.get(CENTRAL_TABLE_CANDIDATE)
if central:
    print("LIKELY CENTRAL TABLE:", CENTRAL_TABLE_CANDIDATE)
    print("-" * 60)
    print(f"Rows: {central['n_rows']:,}")
    print(f"Unique cases (מספר מקרה): {central['df']['מספר מקרה'].nunique():,}")
    print(f"Unique patients: {central['df']['patient'].nunique():,}")
    print(f"Columns: {central['n_cols']}")
    print()
    print("Why this should be the central table:")
    print("  1. Contains the prediction target inputs: surgery start/end timestamps")
    print("  2. Rich case-level context: procedure, diagnosis, department, staff, anesthesia")
    print("  3. Has both case ID (מספר מקרה) and patient ID (patient) for downstream joins")
    print("  4. Largest integrated clinical record per surgery event")
else:
    print(f"Central table candidate '{CENTRAL_TABLE_CANDIDATE}' not found.")


In [ ]:
# Proposed merge order and strategy
merge_strategy = pd.DataFrame([
    {"order": 1, "table": "ניתוחים_260415", "join_key": "מספר מקרה", "join_type": "BASE TABLE", "notes": "Surgery fact table — one row per procedure; deduplicate or aggregate for case-level target"},
    {"order": 2, "table": "הרדמה", "join_key": "מספר מקרה", "join_type": "LEFT JOIN", "notes": "1:1 case attribute — low duplication risk"},
    {"order": 3, "table": "זמן חתימת מנתח על sign in", "join_key": "מספר מקרה", "join_type": "LEFT JOIN", "notes": "Surgeon sign-in timestamp"},
    {"order": 4, "table": "זמן חתימת מנתח על sign out", "join_key": "מספר מקרה", "join_type": "LEFT JOIN", "notes": "Surgeon sign-out timestamp"},
    {"order": 5, "table": "סוג דם גובה משקל BMI", "join_key": "מספר מקרה", "join_type": "LEFT JOIN", "notes": "Pre-op vitals and blood type"},
    {"order": 6, "table": "לחץ דם", "join_key": "מקרה → מספר מקרה", "join_type": "LEFT JOIN", "notes": "Rename מקרה to מספר מקרה before join"},
    {"order": 7, "table": "סטורציה", "join_key": "מקרה → מספר מקרה", "join_type": "LEFT JOIN", "notes": "Rename מקרה to מספר מקרה before join"},
    {"order": 8, "table": "עישון", "join_key": "מספר מקרה", "join_type": "LEFT JOIN", "notes": "Smoking status per case"},
    {"order": 9, "table": "צוות סיעודי בניתוח", "join_key": "מספר מקרה", "join_type": "LEFT JOIN + AGGREGATE", "notes": "Multiple nurses per case — pivot/aggregate to avoid row explosion"},
    {"order": 10, "table": "תרופות בניתוח", "join_key": "מספר מקרה", "join_type": "LEFT JOIN + AGGREGATE", "notes": "319K rows — long format; aggregate or feature-engineer before join"},
    {"order": 11, "table": "בדיקות מעבדה", "join_key": "מספר מקרה", "join_type": "LEFT JOIN + AGGREGATE", "notes": "3.7M rows — pivot lab tests or filter pre-op window; high cardinality"},
    {"order": 12, "table": "מחלות רקע", "join_key": "Patient → patient", "join_type": "LEFT JOIN + AGGREGATE", "notes": "Patient-level ICD codes — aggregate to case via patient ID"},
    {"order": 13, "table": "רגישות לתרופות", "join_key": "Patient → patient", "join_type": "LEFT JOIN + AGGREGATE", "notes": "Patient-level drug allergies"},
    {"order": 14, "table": "רגישות אחרת", "join_key": "Patient → patient", "join_type": "LEFT JOIN + AGGREGATE", "notes": "Patient-level other allergies"},
    {"order": 15, "table": "תרופות קבועות", "join_key": "Patient → patient", "join_type": "LEFT JOIN + AGGREGATE", "notes": "Chronic medications — aggregate before join"},
])

print("RECOMMENDED MERGE ORDER (future ETL)")
display(merge_strategy)


In [ ]:
# Merge risk register
merge_risks = pd.DataFrame([
    {
        "risk": "Row duplication from multi-record tables",
        "affected_tables": "ניתוחים_260415 (multi-procedure), צוות סיעודי, תרופות בניתוח, בדיקות מעבדה",
        "severity": "HIGH",
        "mitigation": "Aggregate to case level before joining; define grain (case vs procedure) explicitly",
    },
    {
        "risk": "Inconsistent case ID column naming",
        "affected_tables": "לחץ דם, סטורציה (מקרה) vs others (מספר מקרה)",
        "severity": "MEDIUM",
        "mitigation": "Standardize column names in ETL staging layer",
    },
    {
        "risk": "Patient-level tables joined on case grain",
        "affected_tables": "מחלות רקע, רגישות לתרופות, רגישות אחרת, תרופות קבועות",
        "severity": "HIGH",
        "mitigation": "Join via patient ID, aggregate chronic conditions/allergies; watch for many-to-many explosion",
    },
    {
        "risk": "Missing / mismatched keys",
        "affected_tables": "All satellite tables",
        "severity": "MEDIUM",
        "mitigation": "Left joins + null rate monitoring; investigate cases in surgeries not in satellites",
    },
    {
        "risk": "Headerless lab CSV",
        "affected_tables": "בדיקות מעבדה",
        "severity": "LOW",
        "mitigation": "Assign column names at load time (done in this notebook)",
    },
    {
        "risk": "Large table memory / performance",
        "affected_tables": "בדיקות מעבדה (3.7M rows), תרופות בניתוח (320K rows)",
        "severity": "MEDIUM",
        "mitigation": "Filter to relevant cases and pre-op window; aggregate before merge",
    },
    {
        "risk": "Ambiguous join for patient ID",
        "affected_tables": "Patient vs Medical Record vs patient columns",
        "severity": "MEDIUM",
        "mitigation": "Validate Patient ID mapping between tables; Medical Record may differ from Patient",
    },
    {
        "risk": "Cartesian product from unaggregated long-format tables",
        "affected_tables": "בדיקות מעבדה × תרופות בניתוח if joined without aggregation",
        "severity": "CRITICAL",
        "mitigation": "Never join two long-format tables directly — aggregate each to case level first",
    },
])

print("MERGE RISK REGISTER")
display(merge_risks)


## 8. Executive SummaryAuto-generated summary based on the inventory analysis above.

In [ ]:
# Build executive summary from analysis results
exec_lines = []
exec_lines.append("=" * 80)
exec_lines.append("EXECUTIVE SUMMARY — Raw Data Inventory")
exec_lines.append("Hillel Yaffe Medical Center | Surgery Duration Prediction")
exec_lines.append("=" * 80)

exec_lines.append("\n1. DATASET OVERVIEW (what each file likely represents)")
exec_lines.append("-" * 60)
for name, info in sorted(datasets.items(), key=lambda x: x[0]):
    exec_lines.append(f"  • {name}: {info['description']} [{info['n_rows']:,} rows]")

exec_lines.append("\n2. CENTRAL TABLE RECOMMENDATION")
exec_lines.append("-" * 60)
exec_lines.append(f"  Central table: {CENTRAL_TABLE_CANDIDATE} (Surgeries)")
exec_lines.append("  Rationale: Contains surgery timestamps (target variable), procedure/diagnosis")
exec_lines.append("  codes, department, staff, and both case + patient identifiers.")

exec_lines.append("\n3. RECOMMENDED JOIN KEYS")
exec_lines.append("-" * 60)
exec_lines.append("  Primary (case-level):  מספר מקרה  (standardize מקרה → מספר מקרה in לחץ דם, סטורציה)")
exec_lines.append("  Secondary (patient-level):  Patient / patient  (for chronic conditions, allergies, home meds)")
exec_lines.append("  Avoid direct join on:  Medical Record  (verify mapping to Patient first)")

exec_lines.append("\n4. KEY RELATIONSHIPS DISCOVERED")
exec_lines.append("-" * 60)
if not relationship_report.empty:
    top_rels = relationship_report.head(10)
    for _, r in top_rels.iterrows():
        exec_lines.append(
            f"  • {r['table_a']}.{r['column_a']} ↔ {r['table_b']}.{r['column_b']}"
            f"  [{r['relationship_type']}] — {r['suggested_relationship']}"
        )
else:
    exec_lines.append("  (Run cross-table analysis cells to populate)")

exec_lines.append("\n5. MERGE RISKS")
exec_lines.append("-" * 60)
for _, r in merge_risks.iterrows():
    exec_lines.append(f"  [{r['severity']}] {r['risk']}")

exec_lines.append("\n6. RECOMMENDED NEXT STEPS FOR ETL PIPELINE")
exec_lines.append("-" * 60)
next_steps = [
    "Define modeling grain: one row per surgery case (מספר מקרה) vs per procedure",
    "Build staging layer: standardize column names (מקרה → מספר מקרה, Patient → patient)",
    "Compute target variable: surgery duration from start/end timestamps in ניתוחים",
    "Aggregate long-format tables (labs, intra-op meds, nursing) to case level",
    "Aggregate patient-level tables (ICD, allergies, chronic meds) before joining",
    "Profile key overlap rates between surgeries and each satellite table",
    "Document null rates and unmatched cases after each staged join",
    "Proceed to 02_staging / 03_feature_engineering notebooks",
]
for i, step in enumerate(next_steps, 1):
    exec_lines.append(f"  {i}. {step}")

exec_lines.append("\n" + "=" * 80)

executive_summary = "\n".join(exec_lines)
print(executive_summary)


---### Notes- This notebook is **read-only** with respect to source files under `data/raw/`.- The lab file `בדיקות מעבדה.csv` has **no header row**; column names were assigned at load time.- Column `מקרה` in some tables is equivalent to `מספר מקרה` in others — standardize during ETL.- Before merging, decide whether the modeling unit is **surgery case** or **procedure** (ניתוחים has multiple procedures per case).